# Intelligent AI Assistant for Company Documents

## Retrieval-Augmented Generation (RAG)

### Project Overview

This project implements an intelligent AI assistant capable of answering questions about a collection of company documents using **Retrieval-Augmented Generation (RAG)**.

Unlike a traditional chatbot that relies only on the knowledge stored in a language model, this assistant retrieves relevant information from a document collection before generating an answer.

The system is designed to be **dataset-independent**: the underlying documents can be replaced without modifying the core RAG architecture.

### Main Objective

The objective is to build a reusable pipeline capable of:

1. Loading PDF documents.
2. Extracting their textual content.
3. Cleaning and preprocessing the extracted text.
4. Splitting documents into searchable chunks.
5. Converting chunks into semantic embeddings.
6. Storing and searching those embeddings.
7. Retrieving the most relevant information for a user question.
8. Providing the retrieved information to a language model.
9. Generating a grounded answer based on the available documents.
10. Providing the sources used to construct the answer.

### High-Level Architecture

**PDF Documents → Text Extraction → Cleaning → Chunking → Embeddings → Vector Store → Retrieval → LLM → Answer + Sources**

### Important Design Principle

The assistant should not depend on a specific company, document collection, or domain.

The document collection is treated as an external knowledge base. Therefore, replacing the contents of the document directory should allow the same system to operate on a different dataset after rebuilding the document index.


## 1. Environment Setup

Before implementing the RAG pipeline, we install the libraries required for document processing, semantic search, and language-model interaction.

The main components are:

* **PyPDF** — extraction of text from PDF documents.
* **Sentence Transformers** — generation of semantic embeddings.
* **NumPy** — numerical operations and similarity calculations.
* **Transformers** — interaction with language models.
* **Gradio** — optional user interface for interacting with the assistant.

The dependencies are installed once and then imported throughout the notebook.


In [2]:
%pip install -q pypdf sentence-transformers transformers accelerate gradio numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Project Configuration

In this section, we define the paths and configuration used throughout
the notebook.

The document directory is intentionally separated from the processing
logic. This allows the dataset to be replaced later without modifying
the RAG pipeline.

In [15]:
import os
import re
import json
import numpy as np

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("Environment ready.")

c:\Users\PC\CERIST\intelligent-rag-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment ready.


## 2. Dataset Discovery

The assistant should automatically detect the available PDF reports.

At this stage, the real company documents may not yet be available.
This is not a problem: we can build and test the complete pipeline
using temporary demonstration documents.

When the real reports become available, they can simply be placed
inside `data/documents/`.

In [16]:
DATA_DIR = Path("../data/documents")

In [17]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Document directory not found: {DATA_DIR.resolve()}"
    )

pdf_files = sorted(DATA_DIR.glob("*.pdf"))

if not pdf_files:
    raise ValueError(
        f"No PDF documents found in {DATA_DIR.resolve()}"
    )

print(f"✓ Dataset directory: {DATA_DIR.resolve()}")
print(f"✓ Number of PDFs: {len(pdf_files)}")

✓ Dataset directory: C:\Users\PC\CERIST\intelligent-rag-assistant\data\documents
✓ Number of PDFs: 3


## 3. PDF Text Extraction

Large language models cannot directly reason over the raw PDF files
in our pipeline.

The first processing stage therefore converts each PDF into text.

We preserve the document name and page number because this metadata
will later allow the assistant to identify the source of its answers.

In [19]:
from pypdf import PdfReader

In [20]:
def extract_pdf_text(pdf_path):
    """
    Extract text from all pages of a PDF.

    Parameters
    ----------
    pdf_path : Path
        Path to the PDF document.

    Returns
    -------
    list[dict]
        One dictionary per page containing the page text
        and source metadata.
    """
    
    reader = PdfReader(pdf_path)
    
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        
        pages.append({
            "source": pdf_path.name,
            "page": page_number,
            "text": text
        })

    return pages

In [21]:
documents = []

for pdf_path in pdf_files:
    pages = extract_pdf_text(pdf_path)
    documents.extend(pages)

print(f"✓ Loaded {len(pdf_files)} PDF(s)")
print(f"✓ Extracted {len(documents)} page(s)")

✓ Loaded 3 PDF(s)
✓ Extracted 3 page(s)


In [22]:
documents[0]

{'source': 'company_overview.pdf',
 'page': 1,
 'text': 'Example Company - Company Overview\nCompany Activities\nExample Company develops software solutions for businesses. Its main activities include backend\ndevelopment, cloud services, data management, cybersecurity and artificial intelligence.\nEmployees\nThe company has 120 employees distributed across engineering, operations, sales, finance and\nmanagement departments.\nHeadquarters\nThe company headquarters are located in Algiers.\n'}

## 4. Text Cleaning

PDF extraction can introduce unnecessary whitespace, line breaks and
other formatting artifacts.

We normalize the extracted text before creating chunks.

The cleaning process should remain conservative: we do not want to
modify the actual meaning of the document.

In [23]:
def clean_text(text):
    """
    Basic PDF text normalization.
    """
    
    # Replace repeated whitespace
    text = re.sub(r"\s+", " ", text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

In [24]:
for document in documents:
    document["text"] = clean_text(document["text"])

In [25]:
documents = [
    document
    for document in documents
    if document["text"]
]

print(f"✓ {len(documents)} non-empty pages remain.")

✓ 3 non-empty pages remain.


In [26]:
for document in documents[:3]:

    print("=" * 80)
    print(f"Source: {document['source']}")
    print(f"Page : {document['page']}")
    print(document["text"][:500])

Source: company_overview.pdf
Page : 1
Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.
Source: financial_report_2025.pdf
Page : 1
Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Source: operations_report_2025.pdf
Page : 1
Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, data

## 5. Document Chunking

A complete PDF page can contain too much information to send to an LLM
at once.

Instead, we divide the extracted text into smaller overlapping chunks.

The overlap is important because information can span the boundary
between two chunks.

For example:

Chunk 1: "The company invested in artificial intelligence and..."
Chunk 2: "...artificial intelligence and cybersecurity projects during 2025."

Without overlap, important context could be lost.

In [27]:
def chunk_text(text, chunk_size=800, overlap=120):
    """
    Split text into overlapping chunks.

    Parameters
    ----------
    text : str
        Text to split.
    chunk_size : int
        Approximate maximum chunk size in characters.
    overlap : int
        Number of characters shared between consecutive chunks.

    Returns
    -------
    list[str]
        Generated text chunks.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []

    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

## 6. Chunk Metadata

Each chunk keeps information about where it came from.

This allows the assistant to provide source attribution such as:

`financial_report_2025.pdf — page 1`

instead of returning an answer with no indication of its origin.

In [28]:
chunks = []

for document in documents:

    page_chunks = chunk_text(
        document["text"],
        chunk_size=800,
        overlap=120
    )

    for chunk_number, chunk in enumerate(page_chunks):

        chunks.append({
            "chunk_id": (
                f"{document['source']}"
                f"_page_{document['page']}"
                f"_chunk_{chunk_number}"
            ),
            "source": document["source"],
            "page": document["page"],
            "chunk_number": chunk_number,
            "text": chunk
        })

In [29]:
print(f"Total chunks: {len(chunks)}")

Total chunks: 3


In [30]:
chunks[0]

{'chunk_id': 'company_overview.pdf_page_1_chunk_0',
 'source': 'company_overview.pdf',
 'page': 1,
 'chunk_number': 0,
 'text': 'Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.'}

In [31]:
for i, chunk in enumerate(chunks[:5]):

    print("=" * 80)
    print(f"Chunk {i}")
    print(f"Source : {chunk['source']}")
    print(f"Page   : {chunk['page']}")
    print(f"ID     : {chunk['chunk_id']}")
    print()
    print(chunk["text"])

Chunk 0
Source : company_overview.pdf
Page   : 1
ID     : company_overview.pdf_page_1_chunk_0

Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.
Chunk 1
Source : financial_report_2025.pdf
Page   : 1
ID     : financial_report_2025.pdf_page_1_chunk_0

Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Chunk 2
Source : operations_report_2025.pdf

## 7. Dataset Statistics

Before building the retrieval system, we inspect basic statistics
about the processed dataset.

These measurements help us detect problems such as:

- empty documents
- unusually short documents
- unexpectedly large documents
- an excessive number of chunks

In [32]:
from collections import Counter

source_counts = Counter(
    chunk["source"]
    for chunk in chunks
)

print("Documents represented in chunks:")
print()

for source, count in source_counts.items():
    print(f"{source}: {count} chunks")

Documents represented in chunks:

company_overview.pdf: 1 chunks
financial_report_2025.pdf: 1 chunks
operations_report_2025.pdf: 1 chunks


In [33]:
chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print()
print("Chunk statistics")
print("----------------")
print(f"Number of chunks : {len(chunk_lengths)}")
print(f"Minimum length   : {min(chunk_lengths)}")
print(f"Maximum length   : {max(chunk_lengths)}")
print(f"Average length   : {np.mean(chunk_lengths):.1f}")


Chunk statistics
----------------
Number of chunks : 3
Minimum length   : 328
Maximum length   : 444
Average length   : 399.7


## 8. Semantic Embeddings

Keyword search looks for exact words.

Semantic search instead represents text as numerical vectors called
**embeddings**.

Texts with similar meanings should have vectors that are close to
each other in the embedding space.

For example:

"How much money did the company make?"

and

"What was the company's revenue?"

may use different words but express a similar concept.

An embedding model converts both sentences into numerical vectors
that can be compared mathematically.

In [34]:
from sentence_transformers import SentenceTransformer

In [35]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

c:\Users\PC\CERIST\intelligent-rag-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3517.

In [36]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

In [37]:
embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.83it/s]


In [38]:
print("Embedding matrix shape:", embeddings.shape)

Embedding matrix shape: (3, 384)


## 9. Semantic Retrieval

When a user asks a question, we perform the following process:

1. Convert the question into an embedding.
2. Compare it with all document chunk embeddings.
3. Calculate a similarity score.
4. Rank the chunks by similarity.
5. Return the most relevant chunks.

These retrieved chunks will later become the context provided to the LLM.

In [39]:
def retrieve(query, k=5):
    """
    Retrieve the k most relevant document chunks for a query.

    Parameters
    ----------
    query : str
        User question.
    k : int
        Number of chunks to retrieve.

    Returns
    -------
    list[dict]
        Retrieved chunks with similarity scores.
    """

    if not query.strip():
        raise ValueError("Query cannot be empty.")

    if k <= 0:
        raise ValueError("k must be greater than 0.")

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    scores = embeddings @ query_embedding

    top_indices = np.argsort(-scores)[:k]

    results = []

    for index in top_indices:

        result = chunks[index].copy()

        result["score"] = float(scores[index])

        results.append(result)

    return results

In [40]:
results = retrieve(
    "What are the company's main activities?",
    k=5
)
for result in results:

    print("=" * 80)
    print(f"Score  : {result['score']:.4f}")
    print(f"Source : {result['source']}")
    print(f"Page   : {result['page']}")
    print()
    print(result["text"])

Score  : 0.7021
Source : company_overview.pdf
Page   : 1

Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.
Score  : 0.5427
Source : operations_report_2025.pdf
Page   : 1

Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.
Score

In [41]:
results = retrieve(
    "How much revenue did the company generate in 2025?",
    k=3
)

for result in results:

    print("=" * 80)
    print(f"Score  : {result['score']:.4f}")
    print(f"Source : {result['source']}")
    print(result["text"])

Score  : 0.7718
Source : financial_report_2025.pdf
Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Score  : 0.3842
Source : operations_report_2025.pdf
Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.
Score  : 0.3788
Source : company_overview.pdf
Example Company - Company Overview Company Activities Example Company develop

In [42]:
results = retrieve(
    "What is the capital of Japan?",
    k=3
)

for result in results:

    print("=" * 80)
    print(f"Score  : {result['score']:.4f}")
    print(f"Source : {result['source']}")
    print(result["text"])

Score  : 0.1049
Source : operations_report_2025.pdf
Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.
Score  : 0.1032
Source : company_overview.pdf
Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company headquarters are located in Algiers.
Score  : 0.0882
Source : fina

## 10. Retrieval Evaluation

Retrieval quality is one of the most important components of a RAG system.

A powerful language model cannot compensate for completely irrelevant
context.

We therefore test the retriever using questions whose answers are known
to exist in our demonstration documents.

For each question, we inspect:

- the retrieved chunks
- their similarity scores
- their source documents
- whether the expected information was retrieved

In [43]:
def display_retrieval_results(query, k=3):
    """
    Display the chunks retrieved for a user query.
    """

    results = retrieve(query, k=k)

    print(f"Query: {query}")
    print()

    for rank, result in enumerate(results, start=1):

        print("=" * 80)
        print(f"Rank  : {rank}")
        print(f"Score : {result['score']:.4f}")
        print(f"Source: {result['source']}")
        print(f"Page  : {result['page']}")
        print()
        print(result["text"])

In [44]:
display_retrieval_results(
    "What was the company's revenue in 2025?",
    k=3
)

Query: What was the company's revenue in 2025?

Rank  : 1
Score : 0.7266
Source: financial_report_2025.pdf
Page  : 1

Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.
Rank  : 2
Score : 0.4042
Source: operations_report_2025.pdf
Page  : 1

Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.
Rank  : 3
Score : 0.3729
Source: comp

## 11. Retrieval Test Set

We define a small collection of questions with known relevant documents.

This gives us a repeatable way to test whether changes to the retrieval
pipeline improve or degrade performance.

In [45]:
evaluation_questions = [
    {
        "question": "What are the company's main activities?",
        "expected_source": "company_overview.pdf"
    },
    {
        "question": "How many employees does the company have?",
        "expected_source": "company_overview.pdf"
    },
    {
        "question": "What was the company's revenue in 2025?",
        "expected_source": "financial_report_2025.pdf"
    },
    {
        "question": "How much did the company invest in AI and cybersecurity?",
        "expected_source": "financial_report_2025.pdf"
    },
    {
        "question": "What cybersecurity measures does the company use?",
        "expected_source": "operations_report_2025.pdf"
    },
]

In [46]:
def evaluate_retrieval(test_set, k=3):
    """
    Measure how often the expected source appears
    among the top-k retrieved chunks.
    """

    correct = 0

    for item in test_set:

        results = retrieve(item["question"], k=k)

        retrieved_sources = {
            result["source"]
            for result in results
        }

        if item["expected_source"] in retrieved_sources:
            correct += 1

    accuracy = correct / len(test_set)

    return accuracy

In [47]:
retrieval_accuracy = evaluate_retrieval(
    evaluation_questions,
    k=3
)

print(f"Top-3 retrieval accuracy: {retrieval_accuracy:.2%}")

Top-3 retrieval accuracy: 100.00%


## 12. Language Model

The retrieval system finds relevant information.

The language model is responsible for transforming that information
into a natural-language response.

The model must not be treated as the company's knowledge database.

Instead:

**Retriever = finds knowledge**

**LLM = generates the response**

The retrieved documents are therefore explicitly provided to the LLM
as context.

In [48]:
from transformers import pipeline

In [49]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [50]:
import torch

DEVICE = 0 if torch.cuda.is_available() else -1

print(
    "CUDA available:",
    torch.cuda.is_available()
)

print(
    "Device:",
    "GPU" if DEVICE == 0 else "CPU"
)

CUDA available: False
Device: CPU


In [51]:
if DEVICE == 0:
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
else:
    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Selected model:", MODEL_NAME)

Selected model: Qwen/Qwen2.5-0.5B-Instruct


In [52]:
generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device=DEVICE
)

c:\Users\PC\CERIST\intelligent-rag-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 440.45it/s]


In [53]:
messages = [
    {
        "role": "user",
        "content": "What are the main activities of Example Company?"
    }
]

response = generator(
    messages,
    max_new_tokens=200,
    do_sample=False
)

print(response)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'What are the main activities of Example Company?'}, {'role': 'assistant', 'content': "I'm sorry, but I need more context to provide an accurate answer. Could you please clarify what specific information or question you're asking about Example Company and its activities? Without this additional detail, it's difficult for me to provide a meaningful response. If you have any questions related to Example Company or their operations, feel free to ask them, and I'll do my best to assist you."}]}]


## 13. Building the Retrieved Context

The retrieved chunks are converted into a structured context that can
be inserted into the LLM prompt.

Each piece of context includes its source and page number.

This metadata will later be used for source attribution.

In [54]:
def build_context(results):
    """
    Convert retrieved chunks into a formatted context string.
    """

    context_parts = []

    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"[Document {i}]\n"
            f"Source: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"Content:\n{result['text']}"
        )

    return "\n\n".join(context_parts)

In [55]:
results = retrieve(
    "What was the company's revenue in 2025?",
    k=3
)

context = build_context(results)

print(context)

[Document 1]
Source: financial_report_2025.pdf
Page: 1
Content:
Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.

[Document 2]
Source: operations_report_2025.pdf
Page: 1
Content:
Example Company - Operations Report 2025 Technology Department The technology department manages backend systems, databases, cloud infrastructure and internal software platforms. Cybersecurity The company uses access control, authentication mechanisms, network monitoring and regular security assessments. Artificial Intelligence The company is developing artificial intelligence solutions for document analysis and business process automation.

[Document 3]
Source: company_overview.pdf
Page: 1
Content:
Example Company - Company Overview C

## 14. RAG Prompt

The prompt explicitly separates:

1. The assistant's role.
2. The retrieved documents.
3. The user's question.
4. The rules governing the answer.

The most important rule is that the assistant must rely on the supplied
documents rather than inventing unsupported information.


In [56]:
SYSTEM_PROMPT = """
You are an AI assistant for a company.

Your task is to answer questions using only the information
provided in the retrieved company documents.

Rules:

1. Use only the supplied documents as factual sources.
2. Do not invent information.
3. Do not use outside knowledge to answer company-specific questions.
4. If the documents do not contain enough information, say:
   "I don't know based on the provided documents."
5. Keep the answer concise and clear.
6. When possible, mention the source document and page number.
"""

In [57]:
def build_rag_prompt(question, context):
    """
    Build the prompt sent to the language model.
    """

    return f"""
{SYSTEM_PROMPT}

Retrieved documents:

{context}

User question:

{question}

Answer using only the retrieved documents.
"""

In [59]:
result = answer_question(
    "What was the company's revenue in 2025?"
)

print(result["answer"])

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




You are an AI assistant for a company.

Your task is to answer questions using only the information
provided in the retrieved company documents.

Rules:

1. Use only the supplied documents as factual sources.
2. Do not invent information.
3. Do not use outside knowledge to answer company-specific questions.
4. If the documents do not contain enough information, say:
   "I don't know based on the provided documents."
5. Keep the answer concise and clear.
6. When possible, mention the source document and page number.


Retrieved documents:

[Document 1]
Source: financial_report_2025.pdf
Page: 1
Content:
Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.

[Document 2]
Source: operations_report_2025.pdf
Page: 1
Con

In [60]:
for source in result["sources"]:

    print(
        f"- {source['source']} "
        f"(page {source['page']}, "
        f"score={source['score']:.3f})"
    )

- financial_report_2025.pdf (page 1, score=0.727)
- operations_report_2025.pdf (page 1, score=0.404)
- company_overview.pdf (page 1, score=0.373)


In [61]:
result = answer_question(
    "What is the company's office in Tokyo?",
    k=3
)

print(result["answer"])

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




You are an AI assistant for a company.

Your task is to answer questions using only the information
provided in the retrieved company documents.

Rules:

1. Use only the supplied documents as factual sources.
2. Do not invent information.
3. Do not use outside knowledge to answer company-specific questions.
4. If the documents do not contain enough information, say:
   "I don't know based on the provided documents."
5. Keep the answer concise and clear.
6. When possible, mention the source document and page number.


Retrieved documents:

[Document 1]
Source: company_overview.pdf
Page: 1
Content:
Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company hea

## 15. Relevance Filtering

A vector search always returns the most similar chunks, even when
none of the chunks are actually relevant to the question.

For example, if the user asks about an office in Tokyo but our
documents contain no information about Tokyo, the retriever will
still return the "least unrelated" chunks.

To reduce this problem, we introduce a similarity threshold.

Only chunks whose similarity score is high enough will be passed
to the language model.

This gives us:

Question
    ↓
Semantic Search
    ↓
Similarity Threshold
    ↓
Relevant Context
    ↓
LLM

In [62]:
RELEVANCE_THRESHOLD = 0.35


def retrieve_relevant(query, k=5, threshold=RELEVANCE_THRESHOLD):
    """
    Retrieve relevant document chunks using semantic similarity.

    Chunks below the similarity threshold are discarded.
    """

    results = retrieve(query, k=k)

    relevant_results = [
        result
        for result in results
        if result["score"] >= threshold
    ]

    return relevant_results

In [63]:
results = retrieve_relevant(
    "What is the company's office in Tokyo?",
    k=5
)

print(f"Relevant chunks found: {len(results)}")

for result in results:
    print(
        f"{result['source']} | "
        f"page {result['page']} | "
        f"score={result['score']:.4f}"
    )

Relevant chunks found: 2
company_overview.pdf | page 1 | score=0.4183
operations_report_2025.pdf | page 1 | score=0.4103


In [64]:
results = retrieve_relevant(
    "What was the company's revenue in 2025?",
    k=5
)

print(f"Relevant chunks found: {len(results)}")

for result in results:
    print(
        f"{result['source']} | "
        f"page {result['page']} | "
        f"score={result['score']:.4f}"
    )

Relevant chunks found: 3
financial_report_2025.pdf | page 1 | score=0.7266
operations_report_2025.pdf | page 1 | score=0.4042
company_overview.pdf | page 1 | score=0.3729


In [65]:
def answer_question(question, k=5, threshold=RELEVANCE_THRESHOLD):
    """
    Complete RAG pipeline with relevance filtering.

    Pipeline:

    Question
        ↓
    Retrieval
        ↓
    Relevance filtering
        ↓
    Context construction
        ↓
    LLM generation
        ↓
    Answer + sources
    """

    results = retrieve_relevant(
        question,
        k=k,
        threshold=threshold
    )

    # No sufficiently relevant information was found
    if not results:
        return {
            "question": question,
            "answer": "I don't know based on the provided documents.",
            "sources": []
        }

    context = build_context(results)

    prompt = build_rag_prompt(
        question,
        context
    )

    response = generator(
        prompt,
        max_new_tokens=250,
        do_sample=False
    )

    generated_text = response[0]["generated_text"]

    return {
        "question": question,
        "answer": generated_text,
        "sources": results
    }

In [66]:
result = answer_question(
    "What was the company's revenue in 2025?"
)

print(result["answer"])

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




You are an AI assistant for a company.

Your task is to answer questions using only the information
provided in the retrieved company documents.

Rules:

1. Use only the supplied documents as factual sources.
2. Do not invent information.
3. Do not use outside knowledge to answer company-specific questions.
4. If the documents do not contain enough information, say:
   "I don't know based on the provided documents."
5. Keep the answer concise and clear.
6. When possible, mention the source document and page number.


Retrieved documents:

[Document 1]
Source: financial_report_2025.pdf
Page: 1
Content:
Example Company - Financial Report 2025 Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year. Operating Expenses Operating expenses reached 3.1 million euros in 2025. Investment The company invested 650,000 euros in artificial intelligence and cybersecurity projects during 2025.

[Document 2]
Source: operations_report_2025.pdf
Page: 1
Con

In [67]:
result = answer_question(
    "What is the company's office in Tokyo?"
)

print(result["answer"])

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




You are an AI assistant for a company.

Your task is to answer questions using only the information
provided in the retrieved company documents.

Rules:

1. Use only the supplied documents as factual sources.
2. Do not invent information.
3. Do not use outside knowledge to answer company-specific questions.
4. If the documents do not contain enough information, say:
   "I don't know based on the provided documents."
5. Keep the answer concise and clear.
6. When possible, mention the source document and page number.


Retrieved documents:

[Document 1]
Source: company_overview.pdf
Page: 1
Content:
Example Company - Company Overview Company Activities Example Company develops software solutions for businesses. Its main activities include backend development, cloud services, data management, cybersecurity and artificial intelligence. Employees The company has 120 employees distributed across engineering, operations, sales, finance and management departments. Headquarters The company hea

## 16. Clean LLM Generation

The text-generation pipeline may return the original prompt together
with the generated answer.

For a user-facing assistant, we only want the newly generated text.

We therefore separate the prompt from the generated continuation.

In [68]:
def generate_answer(prompt, max_new_tokens=250):
    """
    Generate only the new text produced by the language model.
    """

    response = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False
    )

    return response[0]["generated_text"].strip()

In [69]:
test_prompt = build_rag_prompt(
    "What was the company's revenue in 2025?",
    build_context(
        retrieve_relevant(
            "What was the company's revenue in 2025?"
        )
    )
)

answer = generate_answer(test_prompt)

print(answer)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year.


## 17. Source Attribution

A professional RAG assistant should not only provide an answer.

It should also indicate where the information came from.

Each retrieved chunk contains metadata such as:

- document name
- page number
- similarity score

This allows us to provide transparent source references.

In [70]:
def format_sources(results):
    """
    Format retrieved document metadata for display.
    """

    if not results:
        return "No sources found."

    lines = []

    for result in results:
        lines.append(
            f"- {result['source']} — page {result['page']}"
        )

    return "\n".join(lines)

In [71]:
results = retrieve_relevant(
    "What was the company's revenue in 2025?"
)

print(format_sources(results))

- financial_report_2025.pdf — page 1
- operations_report_2025.pdf — page 1
- company_overview.pdf — page 1


In [72]:
def answer_question(
    question,
    k=5,
    threshold=RELEVANCE_THRESHOLD
):
    """
    Complete RAG pipeline.

    Question
        ↓
    Semantic retrieval
        ↓
    Relevance filtering
        ↓
    Context construction
        ↓
    Prompt construction
        ↓
    LLM generation
        ↓
    Answer + sources
    """

    if not question.strip():
        raise ValueError("Question cannot be empty.")

    # 1. Retrieve relevant chunks
    results = retrieve_relevant(
        question,
        k=k,
        threshold=threshold
    )

    # 2. Handle questions with no relevant information
    if not results:
        return {
            "question": question,
            "answer": "I don't know based on the provided documents.",
            "sources": []
        }

    # 3. Build context
    context = build_context(results)

    # 4. Build RAG prompt
    prompt = build_rag_prompt(
        question,
        context
    )

    # 5. Generate answer
    answer = generate_answer(prompt)

    # 6. Return structured result
    return {
        "question": question,
        "answer": answer,
        "sources": results
    }

In [73]:
result = answer_question(
    "What was the company's revenue in 2025?"
)

print("ANSWER")
print("-------")
print(result["answer"])

print()
print("SOURCES")
print("-------")
print(format_sources(result["sources"]))

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER
-------
Revenue Example Company generated a total revenue of 4.8 million euros during the 2025 financial year.

SOURCES
-------
- financial_report_2025.pdf — page 1
- operations_report_2025.pdf — page 1
- company_overview.pdf — page 1


In [74]:
result = answer_question(
    "What is the company's office in Tokyo?"
)

print("ANSWER")
print("-------")
print(result["answer"])

print()
print("SOURCES")
print("-------")
print(format_sources(result["sources"]))

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER
-------
The company's headquarters are located in Algiers. There is no mention of any offices being in Tokyo in the given documents. Therefore, I cannot provide an accurate response regarding the location of the company's headquarters in Tokyo. Based solely on the information provided, I am unable to determine the exact location of the company's headquarters. To accurately answer this question, additional information about the company's locations would be needed. However, I can confirm that the company's headquarters are located in Algiers.

SOURCES
-------
- company_overview.pdf — page 1
- operations_report_2025.pdf — page 1


In [75]:
%pip install -q chromadb

Note: you may need to restart the kernel to use updated packages.


In [1]:
import chromadb

In [2]:
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

print("Chroma database:", CHROMA_DIR.resolve())

NameError: name 'PROJECT_ROOT' is not defined

In [ ]:
collection = chroma_client.get_or_create_collection(
    name="company_documents"
)

print("Collection:", collection.name)

In [11]:
def create_demo_pdf(filename, title, sections, data_dir=None):
    """
    Create a temporary PDF document for testing the RAG pipeline.
    
    Parameters
    ----------
    filename : str
        Name of the PDF file.
    title : str
        Document title.
    sections : list[tuple[str, str]]
        List containing section titles and their text.
    data_dir : str or Path, optional
        Directory where the PDF will be written. Uses DATA_DIR when it
        exists, otherwise defaults to ../data/documents.
    """
    
    from pathlib import Path
    
    if data_dir is None:
        data_dir = globals().get("DATA_DIR", "../data/documents")
    
    output_dir = Path(data_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename

    document = SimpleDocTemplate(
        str(output_path),
        pagesize=A4,
        rightMargin=2 * cm,
        leftMargin=2 * cm,
        topMargin=2 * cm,
        bottomMargin=2 * cm
    )

    styles = getSampleStyleSheet()

    content = [
        Paragraph(title, styles["Title"]),
        Spacer(1, 0.5 * cm)
    ]

    for section_title, section_text in sections:
        content.append(
            Paragraph(section_title, styles["Heading2"])
        )
        content.append(
            Paragraph(section_text, styles["BodyText"])
        )
        content.append(
            Spacer(1, 0.3 * cm)
        )

    document.build(content)

    return output_path

In [12]:
create_demo_pdf(
    "company_overview.pdf",
    "Example Company - Company Overview",
    [
        (
            "Company Activities",
            "Example Company develops software solutions for businesses. "
            "Its main activities include backend development, cloud services, "
            "data management, cybersecurity and artificial intelligence."
        ),
        (
            "Employees",
            "The company has 120 employees distributed across engineering, "
            "operations, sales, finance and management departments."
        ),
        (
            "Headquarters",
            "The company headquarters are located in Algiers."
        )
    ]
)

create_demo_pdf(
    "financial_report_2025.pdf",
    "Example Company - Financial Report 2025",
    [
        (
            "Revenue",
            "Example Company generated a total revenue of 4.8 million euros "
            "during the 2025 financial year."
        ),
        (
            "Operating Expenses",
            "Operating expenses reached 3.1 million euros in 2025."
        ),
        (
            "Investment",
            "The company invested 650,000 euros in artificial intelligence "
            "and cybersecurity projects during 2025."
        )
    ]
)

create_demo_pdf(
    "operations_report_2025.pdf",
    "Example Company - Operations Report 2025",
    [
        (
            "Technology Department",
            "The technology department manages backend systems, databases, "
            "cloud infrastructure and internal software platforms."
        ),
        (
            "Cybersecurity",
            "The company uses access control, authentication mechanisms, "
            "network monitoring and regular security assessments."
        ),
        (
            "Artificial Intelligence",
            "The company is developing artificial intelligence solutions "
            "for document analysis and business process automation."
        )
    ]
)

WindowsPath('../data/documents/operations_report_2025.pdf')